In [4]:
import pandas as pd
import os
import pyarrow
from pathlib import Path

#PROJECT_ROOT = Path(__file__).resolve().parents[1]
PROJECT_ROOT = Path.cwd().resolve().parents[0]
SILVER_DATASETS_DIR = PROJECT_ROOT / 'datasets' / 'silver'
BRONZE_DATASETS_DIR = PROJECT_ROOT / 'datasets' / 'bronze'

In [10]:
## Patch Summary Dimension

patches_desc_df = pd.read_csv(BRONZE_DATASETS_DIR / 'patches/scraped_patch_notes.csv')

#patches_desc_df.info()
#patches_desc_df.head(5)


patches_desc_df['patch_start_date'] = pd.to_datetime(patches_desc_df['patch_date'], format='%m/%d/%Y', errors='coerce')
patches_desc_df = patches_desc_df.sort_values(by='patch_start_date', ascending=True)
patches_desc_df['patch_end_date'] = patches_desc_df['patch_start_date'].shift(-1)

patches_desc_df.head(5)

patches_dimension_dir = SILVER_DATASETS_DIR / 'patches_dimension.parquet'
patches_desc_df[['patch_number', 'patch_start_date', 'patch_end_date', 'patch_url']].to_parquet(patches_dimension_dir, engine='pyarrow', index=False)

In [ ]:
##Champions Changes Dimension

patches = pd.read_parquet(SILVER_DATASETS_DIR/'patches_dimension.parquet', engine='pyarrow')

if not os.path.exists(SILVER_DATASETS_DIR / 'champions_changes_fact.parquet'):
    champions_changes = pd.DataFrame(columns=['patch_number', 'champion_name', 'change_type'])
else:
    champions_changes = pd.read_parquet(SILVER_DATASETS_DIR/'champions_changes_dimension.parquet', engine='pyarrow')

for patch in patches.itertuples():
    if patch.patch_number not in champions_changes['patch_number'].values:
        source_file = BRONZE_DATASETS_DIR / f'patches/patch_highlights_champion_changes_{patch.patch_number}.csv'
        if os.path.exists(source_file):
            source_file_data = pd.read_csv(source_file)
            new_changes = pd.DataFrame(columns=['patch_number', 'champion_name', 'change_type'])
            new_changes['patch_number'] = source_file_data['Patch Number']
            new_changes['champion_name'] = source_file_data['Champion']
            new_changes['change_type'] = source_file_data['Change Type']
            champions_changes = pd.concat([champions_changes, new_changes], ignore_index=True)
        else:
            print(f"File {source_file} does not exist.")

champions_changes.to_parquet(SILVER_DATASETS_DIR / 'champions_changes_dimension.parquet', engine='pyarrow', index=False)


In [ ]:
## Pro Matches Hist

oe_file_dir = BRONZE_DATASETS_DIR / 'oe' / '2018_LoL_esports_match_data_from_OraclesElixir.csv'

oe_df = pd.read_csv(oe_file_dir)
match_summary_columns = ['date','side','position','playername','champion','gamelength','result','kills','deaths','assists',
                        'teamkills','teamdeaths','doublekills','triplekills','quadrakills','pentakills','damagetochampions',
                        'dpm','damageshare','damagetakenperminute','damagemitigatedperminute','damagetotowers','total cs',
                        'killsat10','assistsat10','deathsat10','csat10','opp_killsat10','opp_assistsat10','opp_deathsat10','opp_csat10',
                        'killsat15','assistsat15','deathsat15','csat15','opp_killsat15','opp_assistsat15','opp_deathsat15','opp_csat15',
                        'killsat20','assistsat20','deathsat20','csat20','opp_killsat20','opp_assistsat20','opp_deathsat20','opp_csat20',
                        'killsat25','assistsat25','deathsat25','csat25','opp_killsat25','opp_assistsat25','opp_deathsat25','opp_csat25']
ban_list_columns = ['date', 'gameid', 'side', 'ban1', 'ban2', 'ban3', 'ban4', 'ban5']

oe_df['date'] = pd.to_datetime(oe_df['date'], errors='coerce')
matches_summary_df = (oe_df.loc[:, match_summary_columns].rename(columns={'total cs': 'total_cs'}).copy().dropna(subset=['date','champion']))
matches_summary_df.to_parquet(SILVER_DATASETS_DIR / 'pro_matches_summary.parquet', engine='pyarrow', index=False)

ban_list_df = (oe_df.loc[:, ban_list_columns].copy().dropna(subset=['date','gameid','ban1']).drop_duplicates(subset=['date','gameid','side'], keep='first'))
ban_list_df.to_parquet(SILVER_DATASETS_DIR / 'pro_matches_ban_list.parquet', engine='pyarrow', index=False)


In [ ]:
from matches import get_patch_dates, get_last_match_date
from datetime import datetime

##Get first and last patch dates from the patches dimension
first_available_patch_date, last_available_patch_date = get_patch_dates()

##Get last match date from the procceded matches
last_processed_match_date = get_last_match_date()

years_list = list(range(first_available_patch_date.year, datetime.now().year + 1))

print(f"First available patch date: {first_available_patch_date}")
print(f"Last available patch date: {last_available_patch_date}")
print(f"Last processed match date: {last_processed_match_date}")
print(f"Years list: {years_list}")
print(f"Current year: {datetime.now().year}")

First available patch date: 2019-01-08 00:00:00
Last available patch date: 2026-08-11 00:00:00
Last processed match date: 2018-12-31 09:51:07
Years list: [2019, 2020, 2021, 2022, 2023, 2024, 2025]
Current year: 2026


In [2]:
from matches import upload_oe_matches

result = upload_oe_matches(rewrite=False)
print(f"Upload result: {result}")

c:\Users\Feragon\Documents\Portfolio\lol_patches\transformations\matches.py:48: DtypeWarning: Columns (0: url) have mixed types. Specify dtype option on import or set low_memory=False.
  oe_df = pd.read_csv(source_file_dir)
c:\Users\Feragon\Documents\Portfolio\lol_patches\transformations\matches.py:48: DtypeWarning: Columns (0: url) have mixed types. Specify dtype option on import or set low_memory=False.
  oe_df = pd.read_csv(source_file_dir)
c:\Users\Feragon\Documents\Portfolio\lol_patches\transformations\matches.py:48: DtypeWarning: Columns (0: url) have mixed types. Specify dtype option on import or set low_memory=False.
  oe_df = pd.read_csv(source_file_dir)
c:\Users\Feragon\Documents\Portfolio\lol_patches\transformations\matches.py:48: DtypeWarning: Columns (0: url, 1: split) have mixed types. Specify dtype option on import or set low_memory=False.
  oe_df = pd.read_csv(source_file_dir)
c:\Users\Feragon\Documents\Portfolio\lol_patches\transformations\matches.py:48: DtypeWarning: 

Upload result: True


In [5]:
import duckdb

duckdb_conn = duckdb.connect(database=':memory:')
duckdb_conn.execute(f"CREATE TABLE pro_matches_summary AS SELECT * FROM read_parquet('{SILVER_DATASETS_DIR}/pro_matches_summary.parquet')")
duckdb_conn.execute("SELECT year(date) as match_year, COUNT(*) as match_count FROM pro_matches_summary GROUP BY match_year ORDER BY match_year").fetchall()

[(2019, 81270),
 (2020, 97470),
 (2021, 123020),
 (2022, 125260),
 (2023, 111060),
 (2024, 101920),
 (2025, 100380),
 (2026, 59820)]